# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: машинное обучение, нейросети и практика их применения.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



# Врежь RAG в Harness!

RAG - это память для ваших ИИ-агентов и способ "расширить кругозор" вашей LLM. Сегодня я покажу, как RAG позволяет работать ИИ с информацией подобно человеческой памяти: как извлекать оттуда воспоминания и как добавлять новые. И все это будет не в абстрактных примерах с кодом или без оного, а на реальных примерах с использованием Codex и Qdrant.

Разберём, как интегрировать Retrieval-Augmented Generation (RAG) в Harness так, чтобы LLM могла работать с вашей базой знаний и эффективно использовать внешние источники информации.

Мы пройдём через следующие шаги:
1. Я расскажу как вообще работает RAG.
2. Покажу как создать простейший полнотекстовый и семантический индексы для вашей базы знаний на базе Qdrant.
3. Мы сделаем MCP-сервер для интеграции RAG с Codex.
4. Посмотрим, как LLM использует RAG в реальном времени и как это влияет на качество ответов.
5. Улучшим наш MCP: добавим поддержку обновления базы знаний.

MCP-сервер будет запускаться локально, а LLM будет взаимодействовать с ним для получения информации из базы знаний в реальном времени.

Да, здесь есть некоторые упрощения: 
- не используется чанкинг текстовой информации
- не рассмотрена обработка мультимодальных данных (изображений, аудио и т.д.)
- упрощен вопрос безопасности и разрешений доступа к базе знаний.

Разбирать будем на примере следующего датасета:\
https://github.com/easyise/learn_machine_learn/blob/master/data/rag-projects-24.md

Это информация о вымышленных проектах вымышленной компании "ZhabaDushit, inc."


In [ ]:
%pip install qdrant-client[fastembed] sentencepiece tiktoken

## Подготовка базы знаний

Преобразуем данные в формат, готовый для индексирования


In [ ]:
import numpy as np
import pandas as pd
import os

assert 'HF_TOKEN' in os.environ, "HF_TOKEN is not set in the environment"

In [ ]:
import re
from pathlib import Path

text = Path("data/rag-projects-24.md").read_text(encoding="utf-8")
rows = []

for p in re.split(r"(?m)^---\s*$", text):
    title = re.search(r"(?m)^##\s+(.+)", p)
    if not title:
        continue
    part = re.split(r"\n\n\*\*Статус работы:\*\*\s*", p, maxsplit=1)
    fields = dict(re.findall(r"(?m)^- \*\*(.+?):\*\*\s*(.*)", part[0]))
    last = list(re.finditer(r"(?m)^- \*\*.+?:\*\*.*", part[0]))[-1]
    rows.append({"project_name": title[1], **fields,
                 "description": part[0][last.end():].strip(),
                 "work_status": part[1].strip(), "source_md": p.strip()})

df = pd.DataFrame(rows).rename(columns={
    "Код": "code", "Заказчик": "customer", "Предметная область": "domain",
    "Руководитель": "manager", "Статус": "status", "Дата начала": "start_date",
    "Дата окончания": "end_date",
    "Дата окончания активных работ (приостановка)": "active_work_end_date",
    "Задача": "task"
}).set_index("code")
df

## Гибридный поиск в Qdrant

**Qdrant** - это:
- Векторная БД с открытым исходным кодом
- Поддерживает как dense, так и sparse векторы
- Чаще всего используется в системах семантического поиска и RAG (Retrieval-Augmented Generation)
- Имеет как production-версию, так и дает возможность запускать in-memory или с хранением данных на диске

In [ ]:
%pip install -U FlagEmbedding

In [ ]:
from qdrant_client import QdrantClient, models
from FlagEmbedding import BGEM3FlagModel

client = QdrantClient(path="qdrant_data")
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=False)
collection = "projects"

if client.collection_exists(collection):
    client.delete_collection(collection)


In [ ]:
texts = df[["project_name", "customer", "domain", "manager", "task", "description"]]\
    .fillna("")\
    .agg(". ".join, axis=1)\
    .map(lambda x: "Проект: " + x)\
    .tolist()
texts


In [ ]:
emb = model.encode(texts, return_dense=True, return_sparse=True, return_colbert_vecs=False)
emb


In [ ]:
client.create_collection(
    collection,
    vectors_config={"dense": models.VectorParams(size=1024, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams()}
)

client.upsert(collection, points=[
    models.PointStruct(
        id=int(str(idx).removeprefix("PRJ-")),
        vector={
            "dense": emb["dense_vecs"][i].tolist(),
            "sparse": models.SparseVector(
                indices=[int(k) for k in emb["lexical_weights"][i]],
                values=list(emb["lexical_weights"][i].values())
            )
        },
        payload={"project_code": str(idx), **row.where(row.notna(), None).to_dict()}
    )
    for i, (idx, row) in enumerate(df.iterrows())
])

In [ ]:
# query = "Какие проекты связаны с компьютерным зрением?"
# query = "Какие проекты занимаются прогнозированием финансовых рисков?"
query = "Какие проекты ведёт Любовь Наумова?"

emb = model.encode(
    [query], return_dense=True, return_sparse=True,
    return_colbert_vecs=False
)

sparse = emb["lexical_weights"][0]

results = client.query_points(
    collection_name=collection,
    prefetch=[
        models.Prefetch(query=emb["dense_vecs"][0].tolist(), using="dense", limit=10),
        models.Prefetch(
            query=models.SparseVector(
                indices=[int(k) for k in sparse],
                values=list(sparse.values())
            ),
            using="sparse", limit=10
        )
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
    with_payload=True
).points

for p in results:
    print(f'{p.score:.3f} | {p.payload["project_name"]} | {p.payload["manager"]}')
    print(p.payload["description"][:250], "\n")

In [ ]:
client.close()

## Пишем MCP-сервер для RAG

Используем библиотеку `fastmcp` для создания сервера.

In [ ]:
%pip install -U fastmcp

Исходный код сервера `mcp_rag.py` доступен по ссылке:\
https://github.com/easyise/learn_machine_learn/blob/master/mcp_server/mcp_rag.py